# పాఠం 16 - Microsoft Foundry తో స్కేలబుల్ ఏజెంట్లను ప్రేరేపించడం

ఈ నోటుబుకులో మీరు **Contoso** అనే కల్పిత సంస్థ కోసం **ఉత్పత్తి-సిద్ధ కస్టమర్ సపోర్ట్ ఏజెంట్** ను నిర్మిస్తారు. గత పాఠాల నుండి భిన్నంగా, ఇక్కడ ముఖ్య విషయం ఏజెంట్ యొక్క తర్క చక్రం కాదు — దాని చుట్టూ ఉన్న *అన్ని* విషయం ఏజెంట్‌ను పెద్ద ఎత్తున నడపడానికి సురక్షితంగా చేసే అంశాలు:

1. **పరికరం కాలింగ్** — ఆర్డర్ వెతికే పని మరియు టికెట్ సృష్టి.
2. **RAG** — జ్ఞానార్జన ప్రక్రియలో నిబంధనలకు సమాధానాలు.
3. **మెమరీ** — మార్లు గడిచినా కస్టమర్‌ను గుర్తుంచుకోవడం.
4. **మోడల్ రూటింగ్** — సరళమైన అభ్యర్థనలను చిన్న మోడల్‌కి, సంక్లిష్టమైనవి పెద్ద మోడల్‌కు పంపడం.
5. **ప్రతిస్పందన క్యాచింగ్** — మోడల్ కాల్ లేకుండా పునరావృత ప్రశ్నలకు సేవలు అందించడం.
6. **మానవ ఆమోదం** — నిర్ణీత పరిమితికి మించి రీఫండ్‌లు ఆమోదం కోసం నిలిపివేయడం.
7. **మూల్యాంకన గేట్** — ఒక ఆఫ్‌లైన్ పరీక్షా సెట్, మరోవైపు ఒక నెమ్మదైన విడుదలను నిరోధించడం.
8. **నిరీక్షణ సామర్ధ్యం** — ప్రతి అభ్యర్థన చుట్టూ OpenTelemetry ట్రేసింగ్.

ప్రతి విభాగం స్వతంత్రంగా ఉండు, నడుపు సులభం. ప్రతి పంక్తిని చదవండి — ఉత్పత్య ప్రాథమిక అంశాలు ఉద్దేశపూర్వకంగా చిన్న పరిమాణంలో ఉంచబడ్డాయి.


## సెటప్

ఈ నోట్బుక్‌ను రన్ చేయముందు, మీరు ఈ క్రిందివి కలిగి ఉండాలి:

1. **డిప్లాయ్ చేయబడిన చాట్ మోడల్ (ఉదా: `gpt-4.1-mini`) ఉన్న Microsoft Foundry ప్రాజెక్ట్**.
2. **Azure CLIలో లాగిన్ అయి ఉండాలి** — మీ టెర్మినల్‌లో `az login` ను రన్ చేయండి.
3. **అవసరమైన ఎన్విరాన్మెంట్ వేరియబుల్స్ సెట్ చేయండి:**
   - `AZURE_AI_PROJECT_ENDPOINT` — మీ Microsoft Foundry ప్రాజెక్ట్ ఎండ్‌పాయింట్.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — మీ డిప్లాయ్ చేసిన మోడల్ పేరు.

RAG సెక్షన్ `AZURE_SEARCH_SERVICE_ENDPOINT` మరియు `AZURE_SEARCH_API_KEY` సెట్ అయితే **Azure AI Search** ఉపయోగిస్తుంది, లేకపోతే సెర్చ్ రిసోర్స్ లేకుండా నోట్బుక్ నడచేందుకు ఇన్-మెమరీ సెర్చ్ fallbackగా పనిచేస్తుంది.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. టూల్స్

ప్రొడక్షన్ టూల్స్ నిజమైన వ్యవస్థలపై నిజమైన పని చేస్తాయి. ఇక్కడ మేము సాధారణ Python ఫంక్షన్‌లతో ఒక ఆర్డర్ డేటాబేస్ మరియు ఒక టికెటింగ్ సిస్టమ్‌ను అనుకరించాము. `@tool` డెకరేటర్ వాటిని ఏజెంట్‌కు అందిస్తుంది.

`issue_refund` లో `approval_mode="always_require"` ని ఒక పరిధి కంటే ఎక్కువ రిఫండ్స్ కోసం ఉపయోగించబడుతుంది — ఇది తర్వాత మేము అమలు చేసే మానవ-ఇన్-ది-లూప్ ప్రిమిటివ్.  


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — విధాన జ్ఞాన శిఖరం

విధాన ప్రశ్నలు ("మీ రిటర్న్ విండో ఎంత?") మోడల్ యొక్క జ్ఞాపకానికి బదులు అధికారిక మూలం నుండి సమాధానం ఇవ్వాలి. మేము ఒక చిన్న జ్ఞాన శిఖరాన్ని శోధన సాధనంగా చుట్టివేస్తాము.

ప్రొడక్షన్‌లో ఇది **Azure AI Search**; ఇక్కడ మేము ఇన్-మెమరీ కీవర్డ్ శోధనను అందిస్తాము కాబట్టి నోట్‌బుక్ ఎక్కడైనా పరుగెత్తుతుంది, మరియు వాతావరణ వేరియబుల్స్ ఉన్నపుడు ఆటోమేటిగ్గా Azure AI Search కి మారుతుంది.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. మెమరీ

ఎవరి తో మాట్లాడుతోన్నదో మర్చిపోయిన సపోర్ట్ ఏజెంట్ ఒక చెడ్డ సపోర్ట్ ఏజెంట్. ప్రతి కస్టమర్ కొరకు మేము ఒక చిన్న ప్రొఫైల్ స్టోర్ ఉంచి, ఆ ఏజెంట్ సూచనలలో ఒక సంక్షిప్త సారాంశాన్ని చేర్చుతాము. ఉత్పత్తిలో ఇది ఒక మెమరీ సర్వీస్ (పాఠం 13 చూడండి); ఇక్కడ ఒక డిక్ ఈ నమూనాను కనిపెడుతుంది.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. మోడల్ రౌటింగ్ మరియు స్పందన క్యాచింగ్

ఒకే అభ్యర్థన హ్యాండ్లర్‌లో రెండు ఖర్చు లేవర్లు కనెక్ట్ చేయబడ్డాయి:

- **రౌటింగ్**: ఒక తక్కువ ఖర్చుheuristic వర్గీకర్త నిర్ణయిస్తుంది అభ్యర్థన చిన్న లేదా పెద్ద మోడల్ అవసరమా అని.
- **క్యాచింగ్**: సాధారణీకృత పునరావృత ప్రశ్నలు మోడల్ కాల్ లేకుండా క్యాష్ నుండి నేరుగా అందించబడతాయి.

ఇక్కడ వర్గీకర్త ఉద్దేశపూర్వకంగా సాదారణమైనది. ఉత్పత్తిలో మీరు దాన్ని ట్రాఫిక్‌కు వ్యతిరేకంగా నిర్ధారించవచ్చు మరియు దాన్ని Foundry యొక్క Model Router తో మార్చవచ్చు.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. ఏజెంట్, మానవ ఆమోదం, మరియు పరిశీలన

ఇప్పుడు మేము పై టూల్స్ నుండి ఏజెంట్ను సమీకరించి ప్రతి అభ్యర్థనను OpenTelemetry స్పాన్‌లో చుట్టబెట్టుదాము. `handle_support_request` ఫంక్షన్ ప్రొడక్షన్ అభ్యర్థన హ్యాండ్లర్: క్యాష్ → రూట్ → ట్రేస్ → రన్ → క్యాష్.

మానవ ఆమోదం ఫ్రేమ్‌వర్క్ ద్వారా నిర్వహించబడుతుంది: ఎందుకంటే `issue_refund` కి `approval_mode="always_require"` ఉన్నందున, రన్ ఆగిపోవడమే కాకుండా మనం స్పష్టంగా పరిష్కరించే ఆమోద అభ్యర్థనను ప్రదర్శిస్తుంది.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. అంచనా గేట్

ఇది పాఠం నుండి విడుదల గేట్: ఒక ఆఫ్‌లైన్ టెస్ట్ సెట్ ఏజెంట్‌ను స్కోర్ చేస్తుంది, మరియు పాస్ రేట్ ఒక మెటుకు సాధించినట్లయితేనే డిప్లాయ్‌మెంట్ కొనసాగుతుంది. ఇక్కడ స్కోర్ చేసే విధానం ఒక సింపుల్ కీవర్డ్-ఓవర్లాప్ తనిఖీ మాత్రమే, ఇది నోట్‌బుక్‌ను స్వయంగా పూర్తిగా ఉంచడానికి; ప్రొడక్షన్‌లో మీరు LLM-అజ్జడ్జ్ లేదా ఫ్రేమ్‌వర్క్ ఎవాల్యువేటర్ ఉపయోగిస్తారు (పాఠం 10 చూడండి).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## ఒకటిగా ఉంచడం: ఒక అనుకరణాత్మక విడుదల

క్రింది సెల్ పాఠం వర్ణించే మొత్తం లూప్‌ను చూపిస్తుంది: అంచనా గేట్‌ను నడిపించండి, మరియు అది తీరితే మాత్రమే "ప్రచురించండి". ఇది మీరు Foundry Agent Service కి ఏజెంట్ను సంస్కరణ అప్గ్రేడ్ చేయడానికి CI లో నడిపించే నమూనా.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## సారాంశం

మీరు ప్రతి ఆపరేషన్ సంబంధిత విషయాలను జత చేసే ఉత్పత్తి-సిద్ధమైన కస్టమర్ సపోర్ట్ ఏజెంట్ ని ఏర్పాటు చేశారు:

- **పరికరాలు, RAG, మరియు మెమొరీ** ఏజెంటుకు సామర్థ్యం మరియు నేపథ్యం ఇస్తాయి.
- **మోడల్ రూటింగ్ మరియు క్యాషింగ్** ఆలస్యం మరియు ఖర్చును పరిమితం చేస్తాయి.
- **మానవ ఆమోదం** పెద్ద హానికరమైన చర్యలపై రక్షణగా ఉంటుంది, ఉదాహరణకు పెద్ద రీఫండ్‌లు.
- **అంచనా గేట్** దిగుబడి రేపిన/releases ముందు చెడని విడుదలలను అరికడుతుంది.
- **ట్రేసింగ్** ప్రతి అభ్యర్థనను గమనించదగినదిగా చేస్తుంది.

### సవాలు

ఈ ఏజెంట్ను విస్తరించండి:

1. **బహుళ మోడళ్ళకు మద్దతు ఇవ్వండి** — మూడవ "కాల్పనాత్మక" స్థాయిని జోడించి, ఎస్కలేషన్లు/ఫిర్యాదులను దానికి మార్గదర్శనం చేయండి.
2. **అంచనా గేట్లు జోడించండి** — `TEST_CASES` లో రీఫండ్-ఆమోద పరిస్థితులను చేర్చండి మరియు గేట్ రిగ్రెషన్లను పట్టుకోతోందని నిర్ధారించండి.
3. **ఖర్చు-జ్ఞానం రూటింగ్ జోడించండి** — ఒక్కో అభ్యర్థన కోసం అంచనా ఖర్చును ట్రాక్ చేయండి (చిన్నది vs పెద్దది vs క్యాష్) మరియు మిశ్రమ ప్రశ్నలు ఉన్న బ్యాచ్ తరువాత ఖర్చు నివేదికను ప్రింట్ చేయండి.

తర్వాతి పాఠంలో మీరు విరుద్ధ ప్రయాణం తీసుకుని, Microsoft Foundry Local మరియు Qwen తో ఒక ఏజెంట్ను పూర్తిగా మీ స్వంత యంత్రంపై నడిపిస్తారు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
